# Notebook 1: Data Engineering & Knowledge Graph Construction
## An Explainable KG-Based Framework for Personalized Drug Repurposing for Bangladeshi Lung Cancer Patients

**Purpose:** This notebook fetches all Tier 1 and Tier 2 secondary datasets and constructs a heterogeneous biomedical Knowledge Graph represented as `nodes.csv` and `edges.csv`.

**Designed for:** Kaggle (Internet: ON | CPU/GPU | RAM: 16GB)

---
### Dataset Coverage
| Tier | Dataset | Role |
|---|---|---|
| 1 | Open Targets | Therapeutic knowledge, target-disease associations |
| 1 | STRING v12 | Protein-protein interactions (PPI) |
| 1 | Reactome | Mechanistic pathways |
| 1 | DrugMechDB | XAI validation paths |
| 1 | GDC/TCGA-LUAD+LUSC | Genomic reference cohort |
| 1 | DrugCentral | Approved drug knowledge |
| 1 | GDSC | Drug sensitivity (IC50/AUC) |
| 1 | DepMap | Functional gene dependency |
| 1 | FDA FAERS | Adverse event safety signals |
| 1 | DailyMed | Regulatory drug labels |
| 2 | ClinVar | Variant clinical significance |
| 2 | UniProt | Protein identifier bridge |
| 2 | Gene Ontology | Functional annotations |
| 2 | GTEx Lung | Normal lung tissue expression |
| 2 | AACR GENIE | Real-world genomic validation |

In [ ]:
# ==============================================================
# SECTION 0: Install dependencies & Environment Setup
# ==============================================================
import subprocess
subprocess.run(['pip', 'install', '-q', 'requests', 'pandas', 'numpy', 'tqdm', 'openpyxl'], check=True)

import os, json, time, gzip, re, hashlib, datetime, warnings, io
import requests
import pandas as pd
import numpy as np
from tqdm import tqdm
from io import StringIO, BytesIO

warnings.filterwarnings('ignore')

# Create full directory structure
DIRS = [
    'data/raw/open_targets', 'data/raw/string', 'data/raw/reactome',
    'data/raw/uniprot',      'data/raw/drugmechdb', 'data/raw/tcga',
    'data/raw/drugcentral',  'data/raw/gdsc', 'data/raw/depmap',
    'data/raw/prism',        'data/raw/faers', 'data/raw/dailymed',
    'data/raw/clinvar',      'data/raw/gene_ontology', 'data/raw/gtex',
    'data/raw/genie',        'data/processed', 'data/metadata'
]
for d in DIRS:
    os.makedirs(d, exist_ok=True)

TIMESTAMP = datetime.datetime.now().isoformat()
METADATA_LOG = []  # Track all downloads for reproducibility

def log_metadata(dataset, provider, url, version, n_records, notes=''):
    METADATA_LOG.append({
        'dataset': dataset, 'provider': provider, 'url': url,
        'version': version, 'download_date': TIMESTAMP,
        'n_records': n_records, 'notes': notes
    })

def safe_request(url, method='GET', retries=3, delay=2, **kwargs):
    """Robust HTTP request with retries and backoff."""
    for attempt in range(retries):
        try:
            if method == 'POST':
                r = requests.post(url, timeout=60, **kwargs)
            else:
                r = requests.get(url, timeout=60, **kwargs)
            if r.status_code == 200:
                return r
            print(f'  Attempt {attempt+1}: HTTP {r.status_code} from {url}')
        except Exception as e:
            print(f'  Attempt {attempt+1} failed: {e}')
        time.sleep(delay * (attempt + 1))
    return None

print(f'Environment ready. Timestamp: {TIMESTAMP}')
print('All output directories created.')

In [ ]:
# ==============================================================
# SECTION 0B: Research Configuration
# ==============================================================

CONFIG = {
    'nsclc_mondo_id'        : 'MONDO_0005233',
    'luad_efo'              : 'EFO_0000571',
    'lusc_efo'              : 'EFO_0000708',
    'species_taxon'         : 9606,
    'string_score_threshold': 700,
    'gdc_projects'          : ['TCGA-LUAD', 'TCGA-LUSC'],
    'open_targets_api'      : 'https://api.platform.opentargets.org/api/v4/graphql',
    'string_api'            : 'https://version-12-0.string-db.org/api/json',
    'gdc_api'               : 'https://api.gdc.cancer.gov',
    'openfda_api'           : 'https://api.fda.gov/drug/event.json',
    'uniprot_api'           : 'https://rest.uniprot.org/uniprotkb',
    'cbioportal_api'        : 'https://www.cbioportal.org/api',
}

# Canonical lung cancer driver genes (seed set for PPI & pathway seeding)
SEED_GENES = [
    'EGFR', 'KRAS', 'ALK', 'ROS1', 'MET', 'RET', 'BRAF', 'TP53',
    'STK11', 'KEAP1', 'CDKN2A', 'SMAD4', 'PIK3CA', 'ERBB2', 'FGFR1',
    'FGFR3', 'DDR2', 'NTRK1', 'NTRK2', 'NTRK3', 'NF1', 'PTEN',
    'RB1', 'ARID1A', 'NFE2L2', 'MYC', 'CD274', 'PDCD1', 'CTLA4',
    'VEGFA', 'MDM2', 'ERBB3', 'AKT1', 'MTOR', 'TSC1', 'TSC2',
    'MAP2K1', 'MAP2K2', 'NRAS', 'HRAS', 'IDH1', 'IDH2'
]

# Approved/investigational drugs for lung cancer (for FAERS/DailyMed queries)
LC_DRUGS = [
    'erlotinib', 'gefitinib', 'osimertinib', 'afatinib', 'dacomitinib',
    'crizotinib', 'alectinib', 'ceritinib', 'brigatinib', 'lorlatinib',
    'pembrolizumab', 'nivolumab', 'atezolizumab', 'durvalumab', 'ipilimumab',
    'carboplatin', 'cisplatin', 'paclitaxel', 'pemetrexed', 'docetaxel',
    'bevacizumab', 'ramucirumab', 'dabrafenib', 'trametinib',
    'selpercatinib', 'pralsetinib', 'capmatinib', 'tepotinib',
    'sotorasib', 'adagrasib'
]

print(f'Seed genes: {len(SEED_GENES)}')
print(f'Lung cancer drugs tracked: {len(LC_DRUGS)}')

---
## SECTION 1: Open Targets Platform
**Role:** Therapeutic knowledge, target-disease associations, and evidence scores.

**API:** `https://api.platform.opentargets.org/api/v4/graphql`

**What we fetch:**
- Top NSCLC-associated gene targets with association scores
- Drug-target-disease associations with clinical trial evidence

In [ ]:
# ==============================================================
# SECTION 1: Open Targets Platform
# ==============================================================

OT_API = CONFIG['open_targets_api']

def fetch_ot_disease_targets(disease_id, page_size=500):
    """Fetch all gene-disease associations for a disease from Open Targets."""
    all_rows = []
    page_index = 0
    while True:
        query = """
        query DiseaseTargets($diseaseId: String!, $index: Int!, $size: Int!) {
          disease(efoId: $diseaseId) {
            id name
            associatedTargets(page: {index: $index, size: $size}) {
              count
              rows {
                target {
                  id
                  approvedSymbol
                  approvedName
                  biotype
                  proteinIds { id source }
                }
                score
                datatypeScores { id score }
              }
            }
          }
        }
        """
        variables = {'diseaseId': disease_id, 'index': page_index, 'size': page_size}
        r = safe_request(OT_API, method='POST',
                         json={'query': query, 'variables': variables})
        if r is None:
            print('  Open Targets API unreachable.')
            break
        data = r.json().get('data', {}).get('disease', {})
        rows = data.get('associatedTargets', {}).get('rows', [])
        if not rows:
            break
        all_rows.extend(rows)
        total = data.get('associatedTargets', {}).get('count', 0)
        print(f'  Page {page_index}: fetched {len(all_rows)}/{total} associations')
        if len(all_rows) >= total:
            break
        page_index += 1
        time.sleep(0.5)
    return all_rows

print('Fetching NSCLC target associations from Open Targets...')
ot_rows = fetch_ot_disease_targets(CONFIG['nsclc_mondo_id'])

ot_records = []
for row in ot_rows:
    t = row['target']
    # Extract UniProt ID
    uniprot_id = None
    for pid in (t.get('proteinIds') or []):
        if pid['source'] == 'uniprot_swissprot':
            uniprot_id = pid['id']; break
    if uniprot_id is None:
        for pid in (t.get('proteinIds') or []):
            if 'uniprot' in pid['source'].lower():
                uniprot_id = pid['id']; break
    # Extract evidence breakdown
    dt_scores = {d['id']: round(d['score'], 4) for d in (row.get('datatypeScores') or [])}
    ot_records.append({
        'disease_id'        : CONFIG['nsclc_mondo_id'],
        'disease_name'      : 'Non-small cell lung carcinoma',
        'target_ensembl_id' : t['id'],
        'gene_symbol'       : t['approvedSymbol'],
        'gene_name'         : t['approvedName'],
        'biotype'           : t.get('biotype', ''),
        'uniprot_id'        : uniprot_id,
        'association_score' : round(row['score'], 4),
        'score_genetic_associations'  : dt_scores.get('genetic_association', 0.0),
        'score_somatic_mutations'     : dt_scores.get('somatic_mutation', 0.0),
        'score_known_drugs'           : dt_scores.get('known_drug', 0.0),
        'score_literature'            : dt_scores.get('literature', 0.0),
        'score_rna_expression'        : dt_scores.get('rna_expression', 0.0),
        'score_animal_models'         : dt_scores.get('animal_model', 0.0),
        'score_affected_pathways'     : dt_scores.get('affected_pathway', 0.0),
    })

ot_df = pd.DataFrame(ot_records)
ot_df.to_csv('data/raw/open_targets/nsclc_targets.csv', index=False)
log_metadata('Open Targets', 'EMBL-EBI', OT_API, 'v4 GraphQL (quarterly update)',
             len(ot_df), 'NSCLC MONDO:0005233 associations')
print(f'Saved {len(ot_df)} target associations → data/raw/open_targets/nsclc_targets.csv')
ot_df[['gene_symbol','association_score','uniprot_id']].head(10)

In [ ]:
# Fetch known drugs from Open Targets for NSCLC
print('Fetching known drugs for NSCLC from Open Targets...')

drug_query = """
query KnownDrugs($diseaseId: String!) {
  disease(efoId: $diseaseId) {
    knownDrugs(size: 500) {
      count
      rows {
        drug { id name drugType maximumClinicalTrialPhase isApproved }
        target { id approvedSymbol }
        phase
        status
        mechanismOfAction
        disease { id name }
      }
    }
  }
}
"""

r = safe_request(OT_API, method='POST',
                 json={'query': drug_query, 'variables': {'diseaseId': CONFIG['nsclc_mondo_id']}})

ot_drugs = []
if r:
    drug_rows = r.json().get('data', {}).get('disease', {}).get('knownDrugs', {}).get('rows', [])
    for row in drug_rows:
        ot_drugs.append({
            'chembl_id'               : row['drug']['id'],
            'drug_name'               : row['drug']['name'],
            'drug_type'               : row['drug']['drugType'],
            'max_clinical_phase'      : row['drug']['maximumClinicalTrialPhase'],
            'is_approved'             : row['drug']['isApproved'],
            'target_ensembl_id'       : row['target']['id'],
            'target_gene_symbol'      : row['target']['approvedSymbol'],
            'disease_id'              : row['disease']['id'],
            'clinical_trial_phase'    : row['phase'],
            'clinical_status'         : row['status'],
            'mechanism_of_action'     : row['mechanismOfAction'],
        })

ot_drugs_df = pd.DataFrame(ot_drugs)
ot_drugs_df.to_csv('data/raw/open_targets/nsclc_known_drugs.csv', index=False)
print(f'Saved {len(ot_drugs_df)} drug-target-disease edges → data/raw/open_targets/nsclc_known_drugs.csv')
ot_drugs_df[['drug_name','target_gene_symbol','clinical_trial_phase','is_approved']].head(10)

---
## SECTION 2: STRING v12 — Protein-Protein Interaction Network
**Role:** Biological network layer for multi-hop graph reasoning.

**Filter:** `combined_score >= 700` (high confidence)

**API:** `https://version-12-0.string-db.org/api/json/network`

In [ ]:
# ==============================================================
# SECTION 2: STRING v12 Protein-Protein Interactions
# ==============================================================

STRING_API = CONFIG['string_api']
SCORE_THRESHOLD = CONFIG['string_score_threshold']

# Use gene symbols from Open Targets + seed genes (deduplicated)
ot_gene_symbols = ot_df['gene_symbol'].dropna().tolist()
all_genes = list(set(ot_gene_symbols + SEED_GENES))
print(f'Querying STRING for {len(all_genes)} genes (combined_score >= {SCORE_THRESHOLD})...')

def fetch_string_ppi(gene_list, species=9606, chunk_size=400):
    """Fetch PPI from STRING in chunks to handle API limits."""
    all_interactions = []
    for i in range(0, len(gene_list), chunk_size):
        chunk = gene_list[i:i+chunk_size]
        identifiers = '%0d'.join(chunk)
        r = safe_request(
            f'{STRING_API}/network',
            method='POST',
            data={
                'identifiers'    : identifiers,
                'species'        : species,
                'required_score' : SCORE_THRESHOLD,
                'network_type'   : 'functional',
                'caller_identity': 'bd_lung_cancer_kg_repurposing'
            }
        )
        if r:
            chunk_data = r.json()
            all_interactions.extend(chunk_data)
            print(f'  Chunk {i//chunk_size + 1}: +{len(chunk_data)} interactions')
        time.sleep(1.5)
    return all_interactions

string_raw = fetch_string_ppi(all_genes)

string_records = []
seen_pairs = set()
for item in string_raw:
    gA = item.get('preferredName_A', '')
    gB = item.get('preferredName_B', '')
    pair_key = tuple(sorted([gA, gB]))
    if pair_key in seen_pairs or gA == gB:
        continue
    seen_pairs.add(pair_key)
    string_records.append({
        'protein_a'       : item.get('stringId_A', ''),
        'protein_b'       : item.get('stringId_B', ''),
        'gene_a'          : gA,
        'gene_b'          : gB,
        'combined_score'  : item.get('score', 0),
        'score_experimental'   : item.get('escore', 0),
        'score_database'       : item.get('dscore', 0),
        'score_coexpression'   : item.get('coexpression', 0),
        'score_textmining'     : item.get('tscore', 0),
    })

string_df = pd.DataFrame(string_records)
# Normalise score (STRING returns 0-1000, convert to 0-1)
if not string_df.empty:
    string_df['combined_score_norm'] = string_df['combined_score'] / 1000.0

string_df.to_csv('data/raw/string/ppi_interactions.csv', index=False)
log_metadata('STRING', 'STRING Consortium', STRING_API, 'v12.0', len(string_df),
             f'combined_score >= {SCORE_THRESHOLD}')
print(f'Saved {len(string_df)} unique PPI edges → data/raw/string/ppi_interactions.csv')
string_df.head(5)

---
## SECTION 3: Reactome Pathway Database
**Role:** Mechanistic pathway layer for XAI biological interpretability.

**Data:** gene2Reactome mapping + Reactome pathway hierarchy

**Download:** `https://reactome.org/download/current/gene2Reactome.txt`

In [ ]:
# ==============================================================
# SECTION 3: Reactome Pathway Database
# ==============================================================

print('Downloading Reactome gene2Reactome mapping...')

REACTOME_GENE2PATH_URL = 'https://reactome.org/download/current/gene2Reactome.txt'
REACTOME_PATHWAYS_URL  = 'https://reactome.org/download/current/ReactomePathways.txt'

r_gene2path = safe_request(REACTOME_GENE2PATH_URL)
r_pathways  = safe_request(REACTOME_PATHWAYS_URL)

reactome_df = pd.DataFrame()
pathways_df = pd.DataFrame()

if r_gene2path:
    gene2path = pd.read_csv(
        StringIO(r_gene2path.text), sep='\t', header=None,
        names=['gene_id','reactome_pathway_id','url','pathway_name','evidence','species']
    )
    # Filter human pathways
    reactome_df = gene2path[gene2path['species'] == 'Homo sapiens'].copy()
    # Filter to genes in our KG universe
    # gene_id here is Ensembl gene ID — join with Open Targets data
    ot_ensembl_ids = set(ot_df['target_ensembl_id'].dropna())
    reactome_df_filtered = reactome_df[reactome_df['gene_id'].isin(ot_ensembl_ids)].copy()
    reactome_df_filtered.to_csv('data/raw/reactome/gene2pathway.csv', index=False)
    print(f'gene2Reactome: {len(reactome_df_filtered)} human gene-pathway pairs for our targets')

if r_pathways:
    pathways_df = pd.read_csv(
        StringIO(r_pathways.text), sep='\t', header=None,
        names=['reactome_pathway_id', 'pathway_name', 'species']
    )
    pathways_human = pathways_df[pathways_df['species'] == 'Homo sapiens']
    pathways_human.to_csv('data/raw/reactome/pathways.csv', index=False)
    print(f'Reactome pathways: {len(pathways_human)} human pathways saved')

log_metadata('Reactome', 'EMBL-EBI', REACTOME_GENE2PATH_URL,
             'Current release (see reactome.org/download)', len(reactome_df_filtered),
             'Human gene-pathway associations')

if not reactome_df_filtered.empty:
    reactome_df_filtered[['gene_id','reactome_pathway_id','pathway_name']].head(5)

---
## SECTION 4: UniProt — Protein Identifier Bridge
**Role:** Central identifier mapping between Ensembl, HGNC, UniProt, and STRING.

**API:** `https://rest.uniprot.org/uniprotkb/`

In [ ]:
# ==============================================================
# SECTION 4: UniProt Human Proteome (Lung Cancer Gene Set)
# ==============================================================

UNIPROT_API = CONFIG['uniprot_api']
print('Fetching UniProt protein metadata for lung cancer gene set...')

UNIPROT_FIELDS = (
    'accession,gene_names,protein_name,organism_name,'
    'organism_id,length,mass,cc_function,cc_subcellular_location,'
    'ft_domain,go_id,go_p,go_f,go_c,'
    'xref_ensembl,xref_string,xref_chembl,xref_drugcentral,'
    'reviewed,annotation_score'
)

def fetch_uniprot_gene(gene_symbol):
    url = f'{UNIPROT_API}/stream'
    params = {
        'query'  : f'gene_exact:{gene_symbol} AND organism_id:{CONFIG["species_taxon"]} AND reviewed:true',
        'format' : 'tsv',
        'fields' : UNIPROT_FIELDS,
        'size'   : 5
    }
    r = safe_request(url, params=params)
    if r and r.text.strip():
        try:
            return pd.read_csv(StringIO(r.text), sep='\t')
        except:
            pass
    return pd.DataFrame()

uniprot_genes_to_fetch = list(set(ot_df['gene_symbol'].dropna().tolist() + SEED_GENES))[:300]
uniprot_records = []

for gene in tqdm(uniprot_genes_to_fetch, desc='UniProt'):
    gene_df = fetch_uniprot_gene(gene)
    if not gene_df.empty:
        gene_df['query_gene_symbol'] = gene
        uniprot_records.append(gene_df.iloc[[0]])  # Take top reviewed entry
    time.sleep(0.3)

uniprot_df = pd.DataFrame()
if uniprot_records:
    uniprot_df = pd.concat(uniprot_records, ignore_index=True)
    uniprot_df.to_csv('data/raw/uniprot/protein_metadata.csv', index=False)
    log_metadata('UniProt', 'UniProt Consortium', UNIPROT_API,
                 '2026 release series', len(uniprot_df),
                 'Swiss-Prot reviewed human proteins for lung cancer gene set')
    print(f'Saved {len(uniprot_df)} protein records → data/raw/uniprot/protein_metadata.csv')
else:
    print('UniProt fetch returned no results.')

---
## SECTION 5: DrugMechDB — Mechanistic Drug-Disease Paths
**Role:** XAI validation reference. Provides curated Drug → Target → Biological Process → Disease paths.

**Source:** GitHub: `SuLab/DrugMechDB`

In [ ]:
# ==============================================================
# SECTION 5: DrugMechDB
# ==============================================================

DRUGMECHDB_URL = 'https://raw.githubusercontent.com/SuLab/DrugMechDB/main/indication_paths.json'
print('Downloading DrugMechDB mechanistic paths...')

r = safe_request(DRUGMECHDB_URL)
drugmech_records = []

if r:
    mech_data = r.json()
    print(f'Downloaded {len(mech_data)} indication entries from DrugMechDB')

    # Filter entries relevant to lung cancer or our drug set
    lung_cancer_terms = [
        'lung cancer', 'nsclc', 'non-small cell', 'non small cell',
        'lung neoplasm', 'lung carcinoma', 'adenocarcinoma of lung',
        'squamous cell lung'
    ]
    lc_drug_set = set(d.lower() for d in LC_DRUGS)

    for entry in mech_data:
        drug_name = str(entry.get('drug', '')).lower()
        disease   = str(entry.get('disease', '')).lower()
        is_lc_disease = any(t in disease for t in lung_cancer_terms)
        is_lc_drug    = any(d in drug_name for d in lc_drug_set)

        if is_lc_disease or is_lc_drug:
            links = entry.get('links', [])
            for i, link in enumerate(links):
                drugmech_records.append({
                    'drug_name'          : entry.get('drug', ''),
                    'drug_db_id'         : entry.get('drug_db_id', ''),
                    'disease'            : entry.get('disease', ''),
                    'disease_db_id'      : entry.get('disease_db_id', ''),
                    'path_step'          : i,
                    'source_node'        : link.get('subject', ''),
                    'source_type'        : link.get('subject_type', ''),
                    'relation'           : link.get('predicate', ''),
                    'target_node'        : link.get('object', ''),
                    'target_type'        : link.get('object_type', ''),
                    'ref_url'            : entry.get('reference', ''),
                })

drugmech_df = pd.DataFrame(drugmech_records)
drugmech_df.to_csv('data/raw/drugmechdb/lc_mechanistic_paths.csv', index=False)
log_metadata('DrugMechDB', 'Scripps Research (SuLab)', DRUGMECHDB_URL,
             'Zenodo 8139357 + GitHub main', len(drugmech_df),
             'Filtered for lung cancer diseases and lung cancer drugs')
print(f'Saved {len(drugmech_df)} mechanistic path steps → data/raw/drugmechdb/lc_mechanistic_paths.csv')
drugmech_df[['drug_name','disease','source_node','relation','target_node']].head(5)

---
## SECTION 6: GDC / TCGA-LUAD & TCGA-LUSC
**Role:** Genomic reference cohort and baseline lung cancer molecular landscape.

**API:** `https://api.gdc.cancer.gov/`

**Access:** Open-access Masked Somatic Mutation (MAF) files

In [ ]:
# ==============================================================
# SECTION 6A: GDC/TCGA - Discover MAF Files for LUAD & LUSC
# ==============================================================

GDC_API = CONFIG['gdc_api']
print('Querying GDC for TCGA-LUAD and TCGA-LUSC masked somatic mutation files...')

def gdc_search_files(projects, data_type='Masked Somatic Mutation', data_format='MAF', access='open'):
    payload = {
        'filters': json.dumps({
            'op': 'and',
            'content': [
                {'op': 'in',  'content': {'field': 'cases.project.project_id', 'value': projects}},
                {'op': '=',   'content': {'field': 'data_type',   'value': data_type}},
                {'op': '=',   'content': {'field': 'data_format', 'value': data_format}},
                {'op': '=',   'content': {'field': 'access',      'value': access}},
            ]
        }),
        'fields': 'file_id,file_name,cases.project.project_id,file_size,md5sum',
        'format': 'json',
        'size'  : 200,
    }
    r = safe_request(f'{GDC_API}/files', params=payload)
    if r:
        hits = r.json().get('data', {}).get('hits', [])
        return hits
    return []

gdc_files = gdc_search_files(CONFIG['gdc_projects'])
print(f'Found {len(gdc_files)} MAF files for TCGA-LUAD + TCGA-LUSC')
gdc_files_df = pd.DataFrame([
    {
        'file_id'    : f['file_id'],
        'file_name'  : f['file_name'],
        'file_size'  : f['file_size'],
        'project'    : f['cases'][0]['project']['project_id'] if f.get('cases') else 'UNKNOWN'
    } for f in gdc_files
])
gdc_files_df.to_csv('data/raw/tcga/tcga_maf_file_index.csv', index=False)
print(gdc_files_df.groupby('project')['file_id'].count())

In [ ]:
# ==============================================================
# SECTION 6B: GDC/TCGA - Download & Aggregate Mutations
# Note: We aggregate at gene level to avoid storing patient PII
# ==============================================================

def download_gdc_maf(file_id, project_label):
    """Download a single MAF file from GDC and return a DataFrame."""
    url = f'{GDC_API}/data/{file_id}'
    r = safe_request(url, stream=True)
    if r is None:
        return pd.DataFrame()
    content = r.content
    try:
        if content[:2] == b'\x1f\x8b':  # gzip magic bytes
            with gzip.open(BytesIO(content), 'rt') as f:
                maf_text = f.read()
        else:
            maf_text = content.decode('utf-8')
        # MAF files have comment lines starting with '#'
        lines = [l for l in maf_text.split('\n') if not l.startswith('#')]
        maf_df = pd.read_csv(StringIO('\n'.join(lines)), sep='\t', low_memory=False)
        maf_df['source_project'] = project_label
        return maf_df
    except Exception as e:
        print(f'  Error parsing MAF {file_id}: {e}')
        return pd.DataFrame()

MAF_COLUMNS_KEEP = [
    'Hugo_Symbol', 'Chromosome', 'Start_Position', 'End_Position',
    'Strand', 'Variant_Classification', 'Variant_Type',
    'Reference_Allele', 'Tumor_Seq_Allele2',
    'HGVSc', 'HGVSp', 'HGVSp_Short',
    'Transcript_ID', 'Exon_Number',
    't_depth', 't_ref_count', 't_alt_count',
    'n_depth', 'n_ref_count', 'n_alt_count',
    'SIFT', 'PolyPhen', 'CLIN_SIG',
    'SOMATIC', 'PUBMED', 'source_project'
]

print(f'Downloading MAF files for {len(gdc_files_df)} files...')
print('This step downloads one MAF per file — may take 10-20 minutes on Kaggle.')
print('We download up to 30 files (15 LUAD + 15 LUSC) as a representative sample.')

all_maf_dfs = []
luad_files = gdc_files_df[gdc_files_df['project'] == 'TCGA-LUAD']['file_id'].tolist()[:15]
lusc_files = gdc_files_df[gdc_files_df['project'] == 'TCGA-LUSC']['file_id'].tolist()[:15]

for fid, proj in tqdm([(f, 'TCGA-LUAD') for f in luad_files] +
                      [(f, 'TCGA-LUSC') for f in lusc_files],
                      desc='TCGA MAF download'):
    maf = download_gdc_maf(fid, proj)
    if not maf.empty:
        available_cols = [c for c in MAF_COLUMNS_KEEP if c in maf.columns]
        all_maf_dfs.append(maf[available_cols])
    time.sleep(0.5)

tcga_df = pd.DataFrame()
if all_maf_dfs:
    tcga_df = pd.concat(all_maf_dfs, ignore_index=True)
    tcga_df.to_csv('data/raw/tcga/tcga_luad_lusc_mutations.csv.gz',
                   index=False, compression='gzip')
    log_metadata('GDC/TCGA', 'NCI GDC', GDC_API, 'Open-access MAF (Masked Somatic Mutation)',
                 len(tcga_df),
                 'TCGA-LUAD + TCGA-LUSC, 15 samples each')
    print(f'Saved {len(tcga_df)} mutation records → data/raw/tcga/tcga_luad_lusc_mutations.csv.gz')

# Gene-level mutation frequency summary
if not tcga_df.empty and 'Hugo_Symbol' in tcga_df.columns:
    gene_mut_summary = tcga_df.groupby(['Hugo_Symbol', 'source_project']).size().reset_index(name='mutation_count')
    gene_mut_summary_pivot = gene_mut_summary.pivot(index='Hugo_Symbol', columns='source_project', values='mutation_count').fillna(0)
    gene_mut_summary_pivot.to_csv('data/raw/tcga/gene_mutation_frequency.csv')
    print('Gene mutation frequency summary:')
    print(gene_mut_summary_pivot.sum().to_dict())

---
## SECTION 7: DrugCentral — Approved Drug Knowledge Base
**Role:** Standardizes drug identities, establishes drug-target edges, defines approved compound space.

**Source:** `https://drugcentral.org/download`

In [ ]:
# ==============================================================
# SECTION 7: DrugCentral
# ==============================================================

print('Fetching DrugCentral drug-target interaction data...')

# DrugCentral provides drug-target interactions downloadable as a flat file
# Primary download URL for the drug-target interaction file (TSV)
DRUGCENTRAL_URLS = [
    'https://drugcentral.org/static/download/drug_target_interaction.tsv.gz',
    'https://unmtid-shinyapps.net/download/drugcentral/drug_target_interaction.tsv.gz',
]

dc_df = pd.DataFrame()
for url in DRUGCENTRAL_URLS:
    r = safe_request(url)
    if r and len(r.content) > 1000:
        try:
            dc_df = pd.read_csv(BytesIO(r.content), sep='\t', compression='gzip')
            print(f'Downloaded DrugCentral from {url}: {len(dc_df)} records')
            break
        except Exception as e:
            # Try without gzip
            try:
                dc_df = pd.read_csv(BytesIO(r.content), sep='\t')
                print(f'Downloaded DrugCentral (plain TSV): {len(dc_df)} records')
                break
            except:
                print(f'  Could not parse from {url}: {e}')

if dc_df.empty:
    print('DrugCentral direct download unavailable. Using openFDA + Open Targets drug data as fallback.')
    # Reconstruct minimal DrugCentral-like table from Open Targets drug data
    dc_df = ot_drugs_df[['chembl_id','drug_name','drug_type',
                          'target_gene_symbol','mechanism_of_action',
                          'is_approved']].drop_duplicates()
    dc_df.columns = ['drug_chembl_id','drug_name','drug_type',
                     'gene_symbol','mechanism_of_action','is_approved']
    dc_df['source'] = 'Open_Targets_fallback'

# Filter for human targets if column present
if 'ORGANISM' in dc_df.columns:
    dc_df = dc_df[dc_df['ORGANISM'].str.lower().str.contains('homo sapiens', na=False)]

dc_df.to_csv('data/raw/drugcentral/drug_target_interactions.csv', index=False)
log_metadata('DrugCentral', 'University of New Mexico', 'https://drugcentral.org/download',
             'Current release', len(dc_df), 'Human drug-target interactions')
print(f'Saved {len(dc_df)} drug-target records → data/raw/drugcentral/drug_target_interactions.csv')
dc_df.head(5)

---
## SECTION 8: GDSC — Genomics of Drug Sensitivity in Cancer
**Role:** Experimental drug efficacy validation via IC50 and AUC.

**Source:** `https://www.cancerrxgene.org/downloads/bulk_download`

In [ ]:
# ==============================================================
# SECTION 8: GDSC Drug Sensitivity
# ==============================================================

print('Downloading GDSC2 drug sensitivity data...')

# GDSC2 is the higher-confidence dataset (5-dose, 384-well format)
GDSC_URLS = {
    'GDSC2_sensitivity': [
        'https://cog.sanger.ac.uk/cancerrxgene/GDSC_release8.5/GDSC2_fitted_dose_response_27Oct23.xlsx',
        'https://cog.sanger.ac.uk/cancerrxgene/GDSC_release8.5/GDSC2_fitted_dose_response_24Jul22.xlsx',
    ],
    'cell_lines': [
        'https://cog.sanger.ac.uk/cancerrxgene/GDSC_release8.5/Cell_Lines_Details.xlsx',
    ],
}

LUNG_CANCER_TISSUE_TERMS = ['lung', 'nsclc', 'luad', 'lusc', 'non-small cell lung']

gdsc_df = pd.DataFrame()
for url in GDSC_URLS['GDSC2_sensitivity']:
    r = safe_request(url)
    if r and len(r.content) > 10000:
        try:
            gdsc_raw = pd.read_excel(BytesIO(r.content), engine='openpyxl')
            print(f'GDSC2: {len(gdsc_raw)} records downloaded from {url}')
            # Filter for lung cancer cell lines
            if 'TCGA_DESC' in gdsc_raw.columns:
                gdsc_df = gdsc_raw[gdsc_raw['TCGA_DESC'].str.lower().str.contains(
                    'luad|lusc|lung', na=False)].copy()
            elif 'Cancer Type' in gdsc_raw.columns:
                gdsc_df = gdsc_raw[gdsc_raw['Cancer Type'].str.lower().str.contains(
                    'lung', na=False)].copy()
            else:
                gdsc_df = gdsc_raw  # Keep all if cannot filter
            break
        except Exception as e:
            print(f'  Error reading GDSC: {e}')

if gdsc_df.empty:
    print('GDSC2 download failed. Recording placeholder and continuing.')
else:
    # Standardize column names
    col_map = {
        'CELL_LINE_NAME': 'cell_line_name', 'DRUG_NAME': 'drug_name',
        'DRUG_ID': 'drug_id', 'LN_IC50': 'ln_ic50', 'AUC': 'auc',
        'RMSE': 'rmse', 'TCGA_DESC': 'cancer_type',
        'Cell Line Name': 'cell_line_name', 'Drug Name': 'drug_name',
    }
    gdsc_df = gdsc_df.rename(columns={k: v for k, v in col_map.items() if k in gdsc_df.columns})
    if 'ln_ic50' in gdsc_df.columns:
        gdsc_df['ic50'] = np.exp(gdsc_df['ln_ic50'])
    gdsc_df.to_csv('data/raw/gdsc/gdsc2_lung_sensitivity.csv', index=False)
    log_metadata('GDSC2', 'Sanger Institute', GDSC_URLS['GDSC2_sensitivity'][0],
                 'Release 8.5 (2023)', len(gdsc_df), 'Lung cancer cell lines only')
    print(f'Saved {len(gdsc_df)} GDSC2 records → data/raw/gdsc/gdsc2_lung_sensitivity.csv')
    print('IMPORTANT: Record exact file hash for reproducibility.')
    gdsc_df.head(5)

---
## SECTION 9: DepMap — Functional Gene Dependency
**Role:** CRISPR-based gene essentiality to validate biologically plausible drug targets.

**Source:** `https://depmap.org/portal/data_page/?tab=currentRelease`

In [ ]:
# ==============================================================
# SECTION 9: DepMap Public
# ==============================================================

print('Downloading DepMap data for lung cancer cell lines...')

# DepMap data is distributed via figshare. We use known stable URLs.
# Users should verify the current release at https://depmap.org/portal/data_page/
DEPMAP_URLS = {
    'model_info': [
        'https://figshare.com/ndownloader/files/42478766',   # Model.csv (26Q1)
        'https://depmap.org/portal/api/downloads/files?releasename=DepMap+Public+26Q1&filename=Model.csv'
    ],
    'crispr_dependency': [
        'https://figshare.com/ndownloader/files/42478760',   # CRISPRGeneEffect.csv
    ],
    'somatic_mutations': [
        'https://figshare.com/ndownloader/files/42478763',   # OmicsSomaticMutations.csv
    ]
}

def download_depmap_file(urls, label):
    for url in urls:
        r = safe_request(url, retries=2)
        if r and len(r.content) > 5000:
            try:
                df = pd.read_csv(BytesIO(r.content))
                print(f'  {label}: {df.shape} from {url[:60]}...')
                return df
            except Exception as e:
                print(f'  Could not parse {label}: {e}')
    return pd.DataFrame()

depmap_model_df     = download_depmap_file(DEPMAP_URLS['model_info'], 'Model metadata')
depmap_crispr_df    = download_depmap_file(DEPMAP_URLS['crispr_dependency'], 'CRISPR dependency')
depmap_mutation_df  = download_depmap_file(DEPMAP_URLS['somatic_mutations'], 'Somatic mutations')

# Identify lung cancer cell line IDs
lung_model_ids = set()
if not depmap_model_df.empty:
    tissue_col = next((c for c in depmap_model_df.columns
                       if 'tissue' in c.lower() or 'cancer_type' in c.lower()
                       or 'lineage' in c.lower()), None)
    if tissue_col:
        lung_models = depmap_model_df[
            depmap_model_df[tissue_col].str.lower().str.contains('lung', na=False)
        ]
        id_col = next((c for c in depmap_model_df.columns if 'model_id' in c.lower() or 'depmap_id' in c.lower()), None)
        if id_col:
            lung_model_ids = set(lung_models[id_col])
        lung_models.to_csv('data/raw/depmap/lung_cell_lines_metadata.csv', index=False)
        print(f'Identified {len(lung_model_ids)} lung cancer DepMap models')

# Filter CRISPR dependency for lung cancer
if not depmap_crispr_df.empty and lung_model_ids:
    id_col = depmap_crispr_df.columns[0]  # First col is usually model ID
    crispr_lung = depmap_crispr_df[depmap_crispr_df[id_col].isin(lung_model_ids)]
    # Focus on our seed genes
    seed_cols = [c for c in crispr_lung.columns if any(g in c for g in SEED_GENES)]
    crispr_lung_filt = crispr_lung[[id_col] + seed_cols]
    crispr_lung_filt.to_csv('data/raw/depmap/crispr_gene_effect_lung.csv', index=False)
    log_metadata('DepMap', 'Broad Institute', 'https://depmap.org/portal/',
                 'DepMap Public 26Q1', len(crispr_lung_filt),
                 'CRISPR gene effect, lung cancer models, seed genes')
    print(f'Saved CRISPR dependency: {crispr_lung_filt.shape} → data/raw/depmap/crispr_gene_effect_lung.csv')
else:
    print('DepMap CRISPR filter skipped (data unavailable or no lung models found).')

print('DepMap section complete.')

---
## SECTION 10: PRISM Repurposing — Independent Drug Response Validation
**Role:** Independent cross-validation of GDSC drug sensitivity findings.

**Source:** `https://depmap.org/repurposing/`

In [ ]:
# ==============================================================
# SECTION 10: PRISM Repurposing Dataset
# ==============================================================

print('Downloading PRISM repurposing drug response data...')

PRISM_URLS = [
    'https://figshare.com/ndownloader/files/20237718',   # secondary-screen-replicate-collapsed-logfold-change.csv
    'https://ndownloader.figshare.com/files/20237715',   # primary-screen-replicate-treatment-info.csv
]

prism_df = pd.DataFrame()
for url in PRISM_URLS:
    r = safe_request(url, retries=2)
    if r and len(r.content) > 10000:
        try:
            prism_df = pd.read_csv(BytesIO(r.content), index_col=0)
            print(f'PRISM data: {prism_df.shape} rows from {url[:60]}...')
            break
        except Exception as e:
            print(f'  PRISM parse error: {e}')

if prism_df.empty:
    print('PRISM direct download unavailable. Attempting alternate method...')
    # Alternative: download via DepMap portal API
    r = safe_request('https://depmap.org/repurposing/download/repurposing_samples_20200324.txt')
    if r:
        try:
            prism_df = pd.read_csv(StringIO(r.text), sep='\t')
            print(f'PRISM compound info: {prism_df.shape}')
        except:
            print('PRISM data not accessible in this session. Mark as Tier 2 pending.')

if not prism_df.empty:
    # Filter for lung cancer cell lines if model IDs are available
    if lung_model_ids:
        prism_lung = prism_df[prism_df.index.isin(lung_model_ids)] if not prism_df.empty else prism_df
    else:
        prism_lung = prism_df

    # Filter columns for our drugs of interest
    lc_drug_cols = [c for c in prism_lung.columns
                    if any(d.lower() in c.lower() for d in LC_DRUGS)]
    if lc_drug_cols:
        prism_lung = prism_lung[lc_drug_cols]

    prism_lung.to_csv('data/raw/prism/prism_lung_drug_response.csv')
    log_metadata('PRISM', 'Broad Institute', 'https://depmap.org/repurposing/',
                 'Secondary screen (2020)', len(prism_lung),
                 'Log-fold change viability, lung cancer cell lines')
    print(f'Saved PRISM data: {prism_lung.shape} → data/raw/prism/prism_lung_drug_response.csv')
else:
    print('PRISM section: Data not fetched. Will be incorporated in future runs.')

---
## SECTION 11: FDA FAERS — Adverse Event Safety Signals
**Role:** Post-marketing safety evidence for candidate drug safety-aware ranking.

**API:** openFDA `https://api.fda.gov/drug/event.json`

> **Scientific note:** FAERS is a spontaneous reporting system. Reports do NOT prove causality. We use FAERS as a safety SIGNAL source with disproportionality analysis.

In [ ]:
# ==============================================================
# SECTION 11: FDA FAERS via openFDA API
# ==============================================================

OPENFDA_API = CONFIG['openfda_api']
print('Fetching adverse event signals from FDA FAERS via openFDA...')

def fetch_faers_drug(drug_name, limit=100):
    """Fetch adverse event counts for a specific drug."""
    params = {
        'search' : f'patient.drug.medicinalproduct:"{drug_name}"',
        'count'  : 'patient.reaction.reactionmeddrapt.exact',
        'limit'  : limit,
    }
    r = safe_request(OPENFDA_API, params=params, retries=2)
    if r:
        results = r.json().get('results', [])
        records = []
        for item in results:
            records.append({
                'drug_name'          : drug_name,
                'adverse_event_term' : item.get('term', ''),
                'report_count'       : item.get('count', 0),
            })
        return records
    return []

faers_records = []
for drug in tqdm(LC_DRUGS, desc='FAERS'):
    drug_aes = fetch_faers_drug(drug, limit=50)
    faers_records.extend(drug_aes)
    time.sleep(0.5)

faers_df = pd.DataFrame(faers_records)
if not faers_df.empty:
    # Flag serious lung/respiratory adverse events
    serious_terms = ['pneumonia', 'pneumonitis', 'dyspnea', 'respiratory failure',
                     'pleural effusion', 'pulmonary', 'interstitial lung']
    faers_df['is_respiratory_ae'] = faers_df['adverse_event_term'].str.lower().apply(
        lambda x: any(t in x for t in serious_terms)
    )
    faers_df.to_csv('data/raw/faers/lc_drug_adverse_events.csv', index=False)
    log_metadata('FDA FAERS', 'FDA openFDA', OPENFDA_API,
                 'Quarterly updated (current)', len(faers_df),
                 'Top adverse event MedDRA terms per lung cancer drug')
    print(f'Saved {len(faers_df)} adverse event records → data/raw/faers/lc_drug_adverse_events.csv')
    print(faers_df.groupby('drug_name')['report_count'].sum().sort_values(ascending=False).head(10))

---
## SECTION 12: DailyMed — Regulatory Drug Labels
**Role:** Complement FAERS with official label warnings, contraindications, and adverse reactions.

**API:** `https://dailymed.nlm.nih.gov/dailymed/services/v2/`

In [ ]:
# ==============================================================
# SECTION 12: DailyMed Drug Labels
# ==============================================================

DAILYMED_API = CONFIG.get('dailymed_api', 'https://dailymed.nlm.nih.gov/dailymed/services/v2')
print('Fetching drug label data from DailyMed...')

def fetch_dailymed_spls(drug_name):
    """Fetch structured product labels for a drug name."""
    # Step 1: Search for SPL set ID
    r = safe_request(f'{DAILYMED_API}/spls.json',
                     params={'drug_name': drug_name, 'pagesize': 3})
    if not r:
        return []
    spls = r.json().get('data', [])
    records = []
    for spl in spls[:2]:  # Take first 2 results
        setid = spl.get('setid', '')
        if not setid:
            continue
        # Step 2: Get SPL detail
        r2 = safe_request(f'{DAILYMED_API}/spls/{setid}.json')
        if r2:
            detail = r2.json().get('data', {})
            records.append({
                'drug_name'          : drug_name,
                'spl_setid'          : setid,
                'spl_title'          : detail.get('title', ''),
                'active_ingredients' : str(detail.get('active_ingredient', '')),
                'drug_class'         : str(detail.get('pharm_class', '')),
                'published_date'     : detail.get('published_date', ''),
            })
        time.sleep(0.3)
    return records

dailymed_records = []
for drug in tqdm(LC_DRUGS[:20], desc='DailyMed'):
    spls = fetch_dailymed_spls(drug)
    dailymed_records.extend(spls)
    time.sleep(0.5)

dailymed_df = pd.DataFrame(dailymed_records)
if not dailymed_df.empty:
    dailymed_df.to_csv('data/raw/dailymed/lc_drug_labels.csv', index=False)
    log_metadata('DailyMed', 'NLM', 'https://dailymed.nlm.nih.gov',
                 'Current SPL release', len(dailymed_df), 'Structured product labels')
    print(f'Saved {len(dailymed_df)} SPL records → data/raw/dailymed/lc_drug_labels.csv')
else:
    print('DailyMed returned no results in this session.')

---
## SECTION 13: ClinVar — Variant Clinical Significance
**Role:** Distinguishes pathogenic from benign variants; improves mutation prioritization.

**Source:** NCBI FTP `https://ftp.ncbi.nlm.nih.gov/pub/clinvar/`

In [ ]:
# ==============================================================
# SECTION 13: ClinVar Variant Summary
# ==============================================================

CLINVAR_URL = 'https://ftp.ncbi.nlm.nih.gov/pub/clinvar/tab_delimited/variant_summary.txt.gz'
print('Downloading ClinVar variant_summary.txt.gz ...')
print('Note: This file is ~300MB. Kaggle can handle this.')

clinvar_df = pd.DataFrame()
r = safe_request(CLINVAR_URL, retries=2)

if r and len(r.content) > 100000:
    try:
        with gzip.open(BytesIO(r.content), 'rt', encoding='utf-8') as f:
            # Read in chunks to avoid memory issues
            chunks = []
            chunk_size = 100000
            for chunk in pd.read_csv(f, sep='\t', chunksize=chunk_size,
                                     low_memory=False, on_bad_lines='skip'):
                # Filter to lung cancer relevant genes and human variants
                if 'GeneSymbol' in chunk.columns:
                    lc_mask = chunk['GeneSymbol'].isin(set(SEED_GENES + ot_df['gene_symbol'].tolist()))
                    human_mask = chunk.get('Assembly', pd.Series(['GRCh38'] * len(chunk))) == 'GRCh38'
                    filtered_chunk = chunk[lc_mask & human_mask]
                    if not filtered_chunk.empty:
                        chunks.append(filtered_chunk)
        if chunks:
            clinvar_df = pd.concat(chunks, ignore_index=True)
    except Exception as e:
        print(f'Error reading ClinVar: {e}')

if not clinvar_df.empty:
    # Select key columns
    keep_cols = [
        'AlleleID', 'GeneSymbol', 'ClinicalSignificance',
        'PhenotypeList', 'ReviewStatus', 'NumberSubmitters',
        'Chromosome', 'Start', 'Stop', 'ReferenceAllele', 'AlternateAllele',
        'Cytogenetic', 'HGVS_c', 'HGVS_p', 'RS# (dbSNP)'
    ]
    available = [c for c in keep_cols if c in clinvar_df.columns]
    clinvar_df = clinvar_df[available]
    clinvar_df.to_csv('data/raw/clinvar/clinvar_lc_genes.csv.gz', index=False, compression='gzip')
    log_metadata('ClinVar', 'NCBI', CLINVAR_URL,
                 f'Downloaded {TIMESTAMP[:10]} (monthly release)',
                 len(clinvar_df), 'Filtered for lung cancer gene set, GRCh38')
    print(f'Saved {len(clinvar_df)} ClinVar records → data/raw/clinvar/clinvar_lc_genes.csv.gz')
    print(clinvar_df['ClinicalSignificance'].value_counts().head(8))
else:
    print('ClinVar download failed or returned no results for our gene set.')

---
## SECTION 14: Gene Ontology — Functional Annotations
**Role:** Enriches biological interpretation layer with Molecular Function, Biological Process, Cellular Component.

**Source:** `http://geneontology.org/gene-associations/goa_human.gaf.gz`

In [ ]:
# ==============================================================
# SECTION 14: Gene Ontology Annotations
# ==============================================================

GO_URL = 'http://geneontology.org/gene-associations/goa_human.gaf.gz'
print('Downloading Gene Ontology human annotation file (GOA)...')

go_df = pd.DataFrame()
r = safe_request(GO_URL, retries=2)

if r and len(r.content) > 100000:
    try:
        with gzip.open(BytesIO(r.content), 'rt') as f:
            go_lines = []
            for line in f:
                if line.startswith('!'):
                    continue  # Skip comment lines
                go_lines.append(line.strip())

        go_df = pd.read_csv(
            StringIO('\n'.join(go_lines)), sep='\t', header=None,
            names=[
                'db', 'db_object_id', 'db_object_symbol', 'qualifier',
                'go_id', 'db_reference', 'evidence_code', 'with_from',
                'aspect', 'db_object_name', 'db_object_synonym',
                'db_object_type', 'taxon', 'date', 'assigned_by',
                'annotation_extension', 'gene_product_form_id'
            ], on_bad_lines='skip', low_memory=False
        )

        # Filter to our gene set
        all_kg_genes = set(SEED_GENES + ot_df['gene_symbol'].dropna().tolist())
        go_df_filtered = go_df[go_df['db_object_symbol'].isin(all_kg_genes)].copy()

        # Map aspect codes
        go_df_filtered['ontology_category'] = go_df_filtered['aspect'].map(
            {'P': 'Biological_Process', 'F': 'Molecular_Function', 'C': 'Cellular_Component'}
        )

        keep_cols = ['db_object_symbol','go_id','evidence_code','ontology_category','db_object_name']
        go_df_filtered = go_df_filtered[keep_cols].drop_duplicates()
        go_df_filtered.to_csv('data/raw/gene_ontology/go_annotations_lc.csv', index=False)
        log_metadata('Gene Ontology', 'GO Consortium', GO_URL,
                     f'Downloaded {TIMESTAMP[:10]}', len(go_df_filtered),
                     'GOA human, filtered for lung cancer gene set')
        print(f'Saved {len(go_df_filtered)} GO annotations → data/raw/gene_ontology/go_annotations_lc.csv')
        print(go_df_filtered['ontology_category'].value_counts())
    except Exception as e:
        print(f'Error reading GO file: {e}')
else:
    print('Gene Ontology download failed.')

---
## SECTION 15: GTEx — Normal Lung Tissue Expression
**Role:** Normal-tissue context to distinguish cancer-associated vs. normal lung biology.

**Source:** GTEx Analysis V8 (GRCh38)

In [ ]:
# ==============================================================
# SECTION 15: GTEx Normal Lung Expression
# ==============================================================

GTEX_URL = ('https://storage.googleapis.com/gtex_analysis_v8/rna_seq_data/'
             'GTEx_Analysis_2017-06-05_v8_RNASeQCv1.1.9_gene_median_tpm.gct.gz')

print('Downloading GTEx v8 median TPM data...')
print('(~100MB GCT file — Kaggle handles this fine)')

gtex_lung_df = pd.DataFrame()
r = safe_request(GTEX_URL, retries=2)

if r and len(r.content) > 10000:
    try:
        with gzip.open(BytesIO(r.content), 'rt') as f:
            # GCT format: first 2 lines are metadata
            f.readline()  # Version line
            f.readline()  # Dimensions line
            gtex_df = pd.read_csv(f, sep='\t')

        # Find lung-related columns
        lung_cols = [c for c in gtex_df.columns if 'lung' in c.lower()]
        gene_col  = 'Description' if 'Description' in gtex_df.columns else gtex_df.columns[1]

        if lung_cols:
            gtex_lung_df = gtex_df[[gene_col] + lung_cols].copy()
            gtex_lung_df.columns = ['gene_symbol'] + ['lung_tpm_' + c.replace(' ', '_')
                                                        for c in lung_cols]
            # Filter to our gene set
            all_kg_genes = set(SEED_GENES + ot_df['gene_symbol'].dropna().tolist())
            gtex_lung_df = gtex_lung_df[gtex_lung_df['gene_symbol'].isin(all_kg_genes)]

            gtex_lung_df.to_csv('data/raw/gtex/gtex_lung_tpm.csv', index=False)
            log_metadata('GTEx', 'GTEx Consortium', GTEX_URL,
                         'GTEx Analysis V8 (2017-06-05)', len(gtex_lung_df),
                         'Lung tissue median TPM, filtered for KG gene set')
            print(f'Saved {len(gtex_lung_df)} GTEx lung expression records')
            print(f'Lung tissue columns: {lung_cols}')
        else:
            print('No lung tissue columns found in GTEx data.')
    except Exception as e:
        print(f'Error reading GTEx: {e}')
else:
    print('GTEx download failed.')

---
## SECTION 16: AACR Project GENIE — Real-World Genomic Validation
**Role:** External real-world clinical genomic validation to complement TCGA.

**API:** cBioPortal `https://www.cbioportal.org/api/`

In [ ]:
# ==============================================================
# SECTION 16: AACR Project GENIE via cBioPortal API
# ==============================================================

CBIO_API = CONFIG['cbioportal_api']
print('Fetching AACR GENIE lung cancer genomic data from cBioPortal...')

def cbio_get(endpoint, params=None):
    r = safe_request(f'{CBIO_API}/{endpoint}',
                     headers={'Accept': 'application/json'},
                     params=params)
    if r:
        return r.json()
    return []

# Find GENIE studies
all_studies = cbio_get('studies?pageSize=500')
genie_studies = [s for s in all_studies
                 if 'genie' in s.get('studyId', '').lower()
                 or 'GENIE' in s.get('name', '')]
print(f'Found {len(genie_studies)} GENIE studies: {[s["studyId"] for s in genie_studies]}')

genie_mutations = []
# Use the most recent public GENIE study
genie_study_id = next(
    (s['studyId'] for s in genie_studies if 'public' in s.get('studyId', '').lower()),
    (genie_studies[0]['studyId'] if genie_studies else None)
)

if genie_study_id:
    print(f'Using GENIE study: {genie_study_id}')

    # Get lung cancer sample IDs from GENIE
    samples = cbio_get(f'studies/{genie_study_id}/samples?pageSize=10000')

    # Filter lung cancer samples
    lung_samples = [
        s['sampleId'] for s in samples
        if 'lung' in str(s.get('clinicalData', '')).lower()
        or 'NSCLC' in str(s.get('clinicalData', '')).upper()
        or 'LUAD' in s.get('sampleId', '').upper()
        or 'LUSC' in s.get('sampleId', '').upper()
    ]
    print(f'Identified {len(lung_samples)} lung cancer samples in GENIE')

    # Fetch mutations for our seed genes
    molecular_profiles = cbio_get(f'studies/{genie_study_id}/molecular-profiles')
    mut_profile = next((p['molecularProfileId'] for p in molecular_profiles
                        if 'mutation' in p.get('molecularProfileId', '').lower()), None)

    if mut_profile and lung_samples:
        # Fetch mutations in chunks
        for gene_chunk in [SEED_GENES[:10], SEED_GENES[10:20], SEED_GENES[20:]]:
            payload = {
                'sampleIds'          : lung_samples[:500],  # Limit to 500 samples per request
                'entrezGeneIds'      : [],  # cBioPortal accepts Hugo symbols
                'hugoGeneSymbols'    : gene_chunk,
            }
            r_mut = safe_request(
                f'{CBIO_API}/molecular-profiles/{mut_profile}/mutations/fetch',
                method='POST',
                headers={'Content-Type': 'application/json'},
                json=payload
            )
            if r_mut:
                genie_mutations.extend(r_mut.json())
            time.sleep(0.5)

genie_df = pd.DataFrame()
if genie_mutations:
    genie_df = pd.DataFrame(genie_mutations)
    cols_keep = [c for c in [
        'hugoGeneSymbol', 'sampleId', 'patientId', 'mutationType',
        'proteinChange', 'chr', 'startPosition', 'endPosition',
        'referenceAllele', 'variantAllele', 'tumorAltCount',
        'tumorRefCount', 'tumorDepth', 'ncbiBuild'
    ] if c in genie_df.columns]
    genie_df = genie_df[cols_keep]
    genie_df.to_csv('data/raw/genie/genie_lung_mutations.csv', index=False)
    log_metadata('AACR GENIE', 'AACR / cBioPortal', CBIO_API,
                 genie_study_id, len(genie_df),
                 'Lung cancer mutations for seed genes')
    print(f'Saved {len(genie_df)} GENIE mutations → data/raw/genie/genie_lung_mutations.csv')
    genie_df[['hugoGeneSymbol','mutationType','proteinChange']].value_counts().head(10)
else:
    print('GENIE mutation fetch returned no results in this session.')

---
## SECTION 17: Heterogeneous Knowledge Graph Construction
### Build `nodes.csv` and `edges.csv`

**Node Types:** Disease, Gene, Protein, Drug, Pathway, CellLine, AdverseEvent, BiologicalProcess

**Edge Types:** disease_associates_gene | protein_interacts_protein | gene_participates_pathway | drug_targets_protein | drug_mechanism_path | gene_has_mutation | drug_sensitivity_cellline | gene_dependency_cellline | drug_causes_adverseevent | variant_clinical_significance | gene_has_function | gene_expressed_tissue

In [ ]:
# ==============================================================
# SECTION 17A: Build Master Node Table
# ==============================================================

print('Building unified KG node table...')

node_records = []
node_counter = 0
entity_to_node_id = {}  # Global lookup: canonical_id -> integer node_id

def add_node(canonical_id, node_type, label, source, extra_attrs=None):
    """Add a node to the registry if not already present."""
    global node_counter
    if canonical_id in entity_to_node_id:
        return entity_to_node_id[canonical_id]
    nid = node_counter
    entity_to_node_id[canonical_id] = nid
    record = {
        'node_id'      : nid,
        'canonical_id' : canonical_id,
        'node_type'    : node_type,
        'label'        : label,
        'source'       : source,
    }
    if extra_attrs:
        record.update(extra_attrs)
    node_records.append(record)
    node_counter += 1
    return nid

# 1. Disease Node
add_node('MONDO:0005233', 'Disease', 'Non-small cell lung carcinoma', 'Open Targets',
         {'efo_id': 'EFO_0000571', 'mesh_id': 'D002289'})

# 2. Gene Nodes (from Open Targets)
if not ot_df.empty:
    for _, row in ot_df.iterrows():
        add_node(
            f"ENSG:{row['target_ensembl_id']}",
            'Gene',
            row['gene_symbol'],
            'Open Targets',
            {
                'ensembl_id'    : row['target_ensembl_id'],
                'gene_symbol'   : row['gene_symbol'],
                'gene_name'     : row['gene_name'],
                'biotype'       : row.get('biotype', ''),
                'uniprot_id'    : row.get('uniprot_id', ''),
                'ot_assoc_score': row.get('association_score', 0.0),
            }
        )

# 3. Protein Nodes (from UniProt)
if not uniprot_df.empty:
    accession_col = next((c for c in uniprot_df.columns if 'entry' in c.lower() or
                          'accession' in c.lower()), uniprot_df.columns[0])
    gene_col = next((c for c in uniprot_df.columns if 'gene' in c.lower()), None)
    for _, row in uniprot_df.iterrows():
        acc = str(row.get(accession_col, ''))
        if acc and acc != 'nan':
            add_node(
                f'UniProt:{acc}', 'Protein', acc, 'UniProt',
                {'uniprot_accession': acc,
                 'gene_symbol': str(row.get(gene_col, '')) if gene_col else ''}
            )

# 4. Drug Nodes (from Open Targets known drugs + DrugCentral)
if not ot_drugs_df.empty:
    for _, row in ot_drugs_df.iterrows():
        cid = str(row.get('chembl_id', ''))
        if cid and cid != 'nan':
            add_node(
                f'ChEMBL:{cid}', 'Drug', row.get('drug_name', cid), 'Open Targets',
                {'chembl_id'         : cid,
                 'drug_name'         : row.get('drug_name', ''),
                 'drug_type'         : row.get('drug_type', ''),
                 'max_clinical_phase': row.get('max_clinical_phase', ''),
                 'is_approved'       : row.get('is_approved', False)}
            )

# 5. Pathway Nodes (from Reactome)
if not pathways_df.empty:
    pathways_human_local = pathways_df[pathways_df['species'] == 'Homo sapiens'] if 'species' in pathways_df.columns else pathways_df
    for _, row in pathways_human_local.iterrows():
        pid = str(row.get('reactome_pathway_id', ''))
        if pid and pid != 'nan':
            add_node(
                f'Reactome:{pid}', 'Pathway', row.get('pathway_name', pid), 'Reactome',
                {'reactome_id': pid, 'pathway_name': row.get('pathway_name', '')}
            )

# 6. CellLine Nodes (from GDSC)
if not gdsc_df.empty and 'cell_line_name' in gdsc_df.columns:
    for cl in gdsc_df['cell_line_name'].dropna().unique():
        add_node(f'CellLine:{cl}', 'CellLine', cl, 'GDSC',
                 {'cell_line_name': cl, 'tissue': 'Lung'})

# 7. Adverse Event Nodes (from FAERS)
if not faers_df.empty and 'adverse_event_term' in faers_df.columns:
    for ae in faers_df['adverse_event_term'].dropna().unique():
        add_node(f'AE:{ae.replace(" ","_")}', 'AdverseEvent', ae, 'FDA FAERS',
                 {'meddra_term': ae})

# 8. Biological Process Nodes (from GO)
if not go_df_filtered.empty:
    for _, row in go_df_filtered[['go_id','db_object_name','ontology_category']].drop_duplicates().iterrows():
        goid = str(row['go_id'])
        if goid and goid != 'nan':
            add_node(
                f'GO:{goid}', row.get('ontology_category', 'BiologicalProcess'),
                goid, 'Gene Ontology',
                {'go_id': goid}
            )

nodes_df = pd.DataFrame(node_records)
nodes_df.to_csv('data/processed/nodes.csv', index=False)
print(f'\nNode Table Summary:')
print(nodes_df['node_type'].value_counts())
print(f'Total nodes: {len(nodes_df)}')

In [ ]:
# ==============================================================
# SECTION 17B: Build Master Edge Table
# ==============================================================

print('Building KG edge table...')

edge_records = []

def add_edge(src_canonical, dst_canonical, relation_type, weight=1.0, source='', extra=None):
    """Add a directed edge if both endpoints exist in the node registry."""
    src_id = entity_to_node_id.get(src_canonical)
    dst_id = entity_to_node_id.get(dst_canonical)
    if src_id is None or dst_id is None:
        return  # Skip edges with unmapped endpoints
    record = {
        'src_node_id'  : src_id,
        'dst_node_id'  : dst_id,
        'src_canonical': src_canonical,
        'dst_canonical': dst_canonical,
        'relation_type': relation_type,
        'weight'       : round(float(weight), 6),
        'source'       : source,
    }
    if extra:
        record.update(extra)
    edge_records.append(record)

# EDGE SET 1: disease_associates_gene (Open Targets)
print('Building edges: disease_associates_gene (Open Targets)...')
if not ot_df.empty:
    for _, row in ot_df.iterrows():
        add_edge(
            src_canonical = 'MONDO:0005233',
            dst_canonical = f"ENSG:{row['target_ensembl_id']}",
            relation_type = 'disease_associates_gene',
            weight        = row.get('association_score', 0.0),
            source        = 'Open Targets',
            extra         = {'somatic_mutation_score': row.get('score_somatic_mutations', 0.0),
                             'known_drug_score'      : row.get('score_known_drugs', 0.0)}
        )
print(f'  Edges after Open Targets: {len(edge_records)}')

# EDGE SET 2: protein_interacts_protein (STRING PPI)
print('Building edges: protein_interacts_protein (STRING)...')
if not string_df.empty:
    # Map gene symbols to ENSG IDs
    symbol_to_ensg = dict(zip(ot_df['gene_symbol'], ot_df['target_ensembl_id']))
    for _, row in string_df.iterrows():
        ensg_a = symbol_to_ensg.get(row.get('gene_a', ''))
        ensg_b = symbol_to_ensg.get(row.get('gene_b', ''))
        if ensg_a and ensg_b:
            add_edge(
                src_canonical = f'ENSG:{ensg_a}',
                dst_canonical = f'ENSG:{ensg_b}',
                relation_type = 'protein_interacts_protein',
                weight        = row.get('combined_score_norm', row.get('combined_score', 0) / 1000.0),
                source        = 'STRING v12',
                extra         = {'experimental_score': row.get('score_experimental', 0)}
            )
print(f'  Edges after STRING: {len(edge_records)}')

# EDGE SET 3: gene_participates_pathway (Reactome)
print('Building edges: gene_participates_pathway (Reactome)...')
if 'reactome_df_filtered' in dir() and not reactome_df_filtered.empty:
    for _, row in reactome_df_filtered.iterrows():
        add_edge(
            src_canonical = f"ENSG:{row['gene_id']}",
            dst_canonical = f"Reactome:{row['reactome_pathway_id']}",
            relation_type = 'gene_participates_pathway',
            weight        = 1.0,
            source        = 'Reactome',
            extra         = {'evidence': row.get('evidence', ''),
                             'pathway_name': row.get('pathway_name', '')}
        )
print(f'  Edges after Reactome: {len(edge_records)}')

# EDGE SET 4: drug_targets_gene (Open Targets Known Drugs)
print('Building edges: drug_targets_gene (Open Targets + DrugCentral)...')
if not ot_drugs_df.empty:
    ensg_from_symbol = dict(zip(ot_df['gene_symbol'], ot_df['target_ensembl_id']))
    for _, row in ot_drugs_df.iterrows():
        cid    = str(row.get('chembl_id', ''))
        symbol = row.get('target_gene_symbol', '')
        ensg   = ensg_from_symbol.get(symbol)
        if cid and cid != 'nan' and ensg:
            phase = row.get('clinical_trial_phase', 0) or 0
            add_edge(
                src_canonical = f'ChEMBL:{cid}',
                dst_canonical = f'ENSG:{ensg}',
                relation_type = 'drug_targets_gene',
                weight        = float(phase) / 4.0,  # Normalize phase 0-4 to 0-1
                source        = 'Open Targets',
                extra         = {'mechanism_of_action'  : row.get('mechanism_of_action', ''),
                                 'clinical_trial_phase' : phase,
                                 'is_approved'          : str(row.get('is_approved', ''))}
            )
print(f'  Edges after Drug-Target: {len(edge_records)}')

# EDGE SET 5: drug_causes_adverseevent (FAERS)
print('Building edges: drug_causes_adverseevent (FAERS)...')
if not faers_df.empty:
    drug_name_to_chembl = dict(zip(
        ot_drugs_df['drug_name'].str.lower(),
        ot_drugs_df['chembl_id']
    )) if not ot_drugs_df.empty else {}
    max_count = faers_df['report_count'].max() if len(faers_df) > 0 else 1
    for _, row in faers_df.iterrows():
        cid = drug_name_to_chembl.get(row['drug_name'].lower(), '')
        ae  = f"AE:{row['adverse_event_term'].replace(' ','_')}"
        if cid and ae in entity_to_node_id:
            add_edge(
                src_canonical = f'ChEMBL:{cid}',
                dst_canonical = ae,
                relation_type = 'drug_causes_adverseevent',
                weight        = row['report_count'] / max_count,
                source        = 'FDA FAERS',
                extra         = {'report_count'      : row['report_count'],
                                 'is_respiratory_ae' : row.get('is_respiratory_ae', False)}
            )
print(f'  Edges after FAERS: {len(edge_records)}')

# EDGE SET 6: drug_sensitivity_cellline (GDSC)
print('Building edges: drug_sensitivity_cellline (GDSC)...')
if not gdsc_df.empty:
    drug_col = next((c for c in gdsc_df.columns if 'drug' in c.lower()), None)
    cl_col   = next((c for c in gdsc_df.columns if 'cell_line' in c.lower()), None)
    ic50_col = next((c for c in gdsc_df.columns if 'ic50' in c.lower() and 'ln' not in c.lower()), None)
    auc_col  = next((c for c in gdsc_df.columns if 'auc' in c.lower()), None)

    if drug_col and cl_col:
        for _, row in gdsc_df.iterrows():
            drug_name = str(row.get(drug_col, '')).lower()
            cl_name   = str(row.get(cl_col, ''))
            cid       = drug_name_to_chembl.get(drug_name, '') if 'drug_name_to_chembl' in dir() else ''
            if cid and cl_name:
                ic50_val = row.get(ic50_col, None) if ic50_col else None
                auc_val  = row.get(auc_col, None)  if auc_col  else None
                add_edge(
                    src_canonical = f'ChEMBL:{cid}',
                    dst_canonical = f'CellLine:{cl_name}',
                    relation_type = 'drug_sensitivity_cellline',
                    weight        = float(auc_val) if auc_val and not pd.isna(auc_val) else 0.5,
                    source        = 'GDSC2',
                    extra         = {'ic50': ic50_val, 'auc': auc_val}
                )
print(f'  Edges after GDSC: {len(edge_records)}')

# EDGE SET 7: gene_has_function (Gene Ontology)
print('Building edges: gene_has_function (Gene Ontology)...')
if 'go_df_filtered' in dir() and not go_df_filtered.empty:
    ensg_from_symbol = {sym: eid for sym, eid in zip(ot_df['gene_symbol'], ot_df['target_ensembl_id'])}
    for _, row in go_df_filtered.iterrows():
        gene_sym = row.get('db_object_symbol', '')
        ensg     = ensg_from_symbol.get(gene_sym)
        go_id    = str(row.get('go_id', ''))
        if ensg and go_id and f'GO:{go_id}' in entity_to_node_id:
            rel_type = {
                'Biological_Process': 'gene_involved_in_process',
                'Molecular_Function': 'gene_enables_function',
                'Cellular_Component': 'gene_located_in_component',
            }.get(row.get('ontology_category', ''), 'gene_has_function')
            add_edge(
                src_canonical = f'ENSG:{ensg}',
                dst_canonical = f'GO:{go_id}',
                relation_type = rel_type,
                weight        = 1.0,
                source        = 'Gene Ontology',
                extra         = {'evidence_code': row.get('evidence_code', '')}
            )
print(f'  Edges after GO: {len(edge_records)}')

# EDGE SET 8: DrugMechDB mechanistic paths
print('Building edges: drug_mechanism_path (DrugMechDB)...')
if not drugmech_df.empty:
    for _, row in drugmech_df.iterrows():
        src = str(row.get('source_node', '')).strip()
        dst = str(row.get('target_node', '')).strip()
        rel = str(row.get('relation', 'drug_mechanism_path'))
        drug_raw  = str(row.get('drug_db_id', ''))
        drug_canon = f'ChEMBL:{drug_raw}' if 'CHEMBL' in drug_raw.upper() else f'Drug:{drug_raw}'
        if drug_canon in entity_to_node_id:
            add_edge(
                src_canonical = drug_canon,
                dst_canonical = f'Node:{dst.replace(" ","_")}',
                relation_type = f'mech_{rel}',
                weight        = 1.0,
                source        = 'DrugMechDB'
            )
print(f'  Edges after DrugMechDB: {len(edge_records)}')

edges_df = pd.DataFrame(edge_records)
edges_df.to_csv('data/processed/edges.csv', index=False)
print(f'\nEdge Table Summary:')
print(edges_df['relation_type'].value_counts())
print(f'Total edges: {len(edges_df)}')

---
## SECTION 18: Summary Statistics & Reproducibility Metadata

In [ ]:
# ==============================================================
# SECTION 18: Summary, Metadata Logging & Final Verification
# ==============================================================

print('=' * 70)
print('KNOWLEDGE GRAPH CONSTRUCTION COMPLETE')
print('=' * 70)

# Reload from disk and compute final stats
final_nodes = pd.read_csv('data/processed/nodes.csv')
final_edges = pd.read_csv('data/processed/edges.csv')

print(f'\nNodes: {len(final_nodes)}')
print(final_nodes['node_type'].value_counts().to_string())

print(f'\nEdges: {len(final_edges)}')
print(final_edges['relation_type'].value_counts().to_string())

# Save metadata log
metadata_df = pd.DataFrame(METADATA_LOG)
metadata_df.to_csv('data/metadata/dataset_provenance_log.csv', index=False)
print(f'\nDataset provenance log: {len(metadata_df)} datasets recorded')

# Compute SHA-256 hashes for reproducibility
print('\nFile hashes (SHA-256) for reproducibility:')
for fpath in ['data/processed/nodes.csv', 'data/processed/edges.csv']:
    try:
        with open(fpath, 'rb') as f:
            digest = hashlib.sha256(f.read()).hexdigest()
        size_mb = os.path.getsize(fpath) / 1024**2
        print(f'  {fpath}: sha256={digest[:16]}... ({size_mb:.2f} MB)')
    except FileNotFoundError:
        print(f'  {fpath}: NOT FOUND')

print(f'\nTimestamp: {TIMESTAMP}')
print('Ready for Notebook 2: R-GCN Model Training')

In [ ]:
# ==============================================================
# SECTION 18B: Basic KG Validation Checks
# ==============================================================

print('Running KG validation checks...')

checks_passed = 0
checks_total  = 0

def check(condition, message):
    global checks_passed, checks_total
    checks_total += 1
    status = 'PASS' if condition else 'FAIL'
    if condition:
        checks_passed += 1
    print(f'  [{status}] {message}')
    return condition

# Node integrity
check(len(final_nodes) > 0,           'Node table is not empty')
check('node_id' in final_nodes.columns,   'node_id column exists')
check('node_type' in final_nodes.columns, 'node_type column exists')
check(final_nodes['node_id'].nunique() == len(final_nodes), 'All node_ids are unique')
check('Gene' in final_nodes['node_type'].values,    'Gene nodes present')
check('Drug' in final_nodes['node_type'].values,    'Drug nodes present')
check('Disease' in final_nodes['node_type'].values, 'Disease nodes present')
check('Pathway' in final_nodes['node_type'].values, 'Pathway nodes present')

# Edge integrity
check(len(final_edges) > 0,                 'Edge table is not empty')
check('src_node_id' in final_edges.columns, 'src_node_id column exists')
check('dst_node_id' in final_edges.columns, 'dst_node_id column exists')
check('relation_type' in final_edges.columns, 'relation_type column exists')
check('weight' in final_edges.columns,      'edge weight column exists')
check(final_edges['weight'].between(0, 1, inclusive='both').all(), 'All weights in [0, 1]')

# Connectivity check: all edge endpoints in node table
node_id_set = set(final_nodes['node_id'])
src_valid   = final_edges['src_node_id'].isin(node_id_set).all()
dst_valid   = final_edges['dst_node_id'].isin(node_id_set).all()
check(src_valid, 'All source endpoints exist in node table')
check(dst_valid, 'All destination endpoints exist in node table')

# Drug-target edges exist
check('drug_targets_gene' in final_edges['relation_type'].values,
      'Drug-target edges present (drug_targets_gene)')
check('protein_interacts_protein' in final_edges['relation_type'].values,
      'PPI edges present (protein_interacts_protein)')
check('disease_associates_gene' in final_edges['relation_type'].values,
      'Disease-gene edges present (disease_associates_gene)')

print(f'\nValidation result: {checks_passed}/{checks_total} checks passed')
if checks_passed == checks_total:
    print('All checks passed. KG is ready for model training.')
else:
    print('Some checks failed. Review data fetching sections above.')

print('\nNext step: Run Notebook 2 (R-GCN Model Training) using nodes.csv and edges.csv')